In [ ]:
import pandas as pd
import numpy as np

DATA_PATH = "target_match_dataset.csv"

df = pd.read_csv(DATA_PATH)
print(f"数据集读取完成：{df.shape[0]:,} 行 × {df.shape[1]} 列")
print(f"总对局数：{df['match_id'].nunique():,}")

## Step 2：数据质量检查

对 `target_match_dataset.csv` 进行三项基础检查，确保数据在进入 Step 3 分析之前是干净可信的：

| 检查项 | 预期 | 异常说明 |
|--------|------|----------|
| 重复对局 | 每个 match_id 恰好出现 10 次 | 超过 10 次说明同一对局被采集了多次 |
| 玩家缺失 | 每场比赛 5v5 = 10 名玩家 | 少于 10 名说明 API 返回了不完整数据 |
| 字段缺失 | 关键列均无 NaN | 缺失值会导致后续指标计算出错 |

## 1. 重复对局检查

每场 Valorant 竞技赛 5v5，正常情况下同一 `match_id` 应恰好出现 **10 次**（对应 10 名玩家）。

如果某个 `match_id` 出现超过 10 次，说明该场比赛被重复采集了，需要在后续分析中去重处理。

In [ ]:
def check_duplicates(df):
    """
    统计每个 match_id 对应的玩家行数。
    正常值 = 10；超过 10 = 重复采集。
    """
    counts = df.groupby("match_id").size()
    dup    = counts[counts > 10]

    print(f"总对局数：{len(counts):,}")
    if len(dup) == 0:
        print("✅ 无重复对局")
    else:
        print(f"⚠ 发现 {len(dup)} 个重复对局（该 match_id 出现超过 10 次）：")
        print(dup.to_string())
    return dup

dup = check_duplicates(df)

## 2. 玩家缺失检查

正常一场 5v5 应有恰好 **10 名玩家**的数据。

玩家数 < 10 说明 API 在返回该对局时遗漏了部分玩家数据（可能是服务端问题）。
这类不完整对局建议在后续分析中排除，因为缺少完整队伍信息会影响 carrier/loafer 的识别。

In [ ]:
def check_incomplete_matches(df):
    """
    统计玩家数 < 10 的对局。
    同时展示 '8人局'、'9人局' 等分布情况。
    """
    counts     = df.groupby("match_id").size()
    incomplete = counts[counts < 10]

    if len(incomplete) == 0:
        print("✅ 所有对局均有 10 名玩家")
    else:
        print(f"⚠ 发现 {len(incomplete)} 个不完整对局")
        dist = incomplete.value_counts().sort_index()
        dist.index.name = "玩家数"
        dist.name = "对局数"
        print(dist.to_frame().to_string())
        print(f"\n这些对局共涉及 {incomplete.sum()} 行数据")
    return incomplete

incomplete = check_incomplete_matches(df)

## 3. 关键字段缺失检查

Step 3 的分析需要依赖一系列关键字段（ACS、ADR、技能使用次数等）。
如果这些字段存在 `NaN`，会导致指标计算出错或产生偏差，需要提前发现。

以下检查的字段包括：对局标识、比赛基础信息、绩效指标、懈怠相关字段、技能使用字段。

In [ ]:
def check_missing_values(df):
    """
    检查 Step 3 分析所需关键字段的缺失情况。
    输出每个字段的缺失数量和缺失率。
    """
    key_cols = [
        # 对局 / 玩家标识
        "match_id", "puuid", "region", "team", "agent",
        # 比赛基础
        "rounds_played",
        # 绩效（用于计算 ACS / ADR / KAST）
        "score", "kills", "deaths", "assists", "damage_made",
        # 懈怠相关
        "afk_rounds", "rounds_in_spawn", "ff_outgoing",
        # 技能使用（用于技能维度 flag）
        "c_cast", "q_cast", "e_cast", "x_cast"
    ]
    missing = df[key_cols].isnull().sum()
    missing = missing[missing > 0]

    if missing.empty:
        print("✅ 所有关键字段均无缺失值")
    else:
        pct    = (missing / len(df) * 100).round(2)
        result = pd.DataFrame({"缺失数": missing, "缺失率%": pct})
        print(f"⚠ 以下 {len(missing)} 个字段存在缺失值：")
        print(result.to_string())
    return missing

missing = check_missing_values(df)

## 汇总报告

In [ ]:
print("=" * 50)
print("数据质量检查汇总")
print("=" * 50)

print(f"\n总行数：    {len(df):,}")
print(f"总对局数：  {df['match_id'].nunique():,}")
print(f"\n各地区对局数分布：")
region_dist = df.drop_duplicates("match_id")["region"].value_counts()
for r, c in region_dist.items():
    print(f"  {r:6s}: {c}")

print(f"\n重复对局：       {len(dup)}")
print(f"不完整对局：     {len(incomplete)}")
print(f"有缺失的关键字段：{len(missing)}")

print("\n" + "-" * 50)
all_ok = len(dup) == 0 and len(incomplete) == 0 and len(missing) == 0
if all_ok:
    print("✅ 全部检查通过，数据质量良好，可进入 Step 3 分析")
else:
    issues = []
    if len(dup)        > 0: issues.append(f"重复对局 {len(dup)} 个")
    if len(incomplete) > 0: issues.append(f"不完整对局 {len(incomplete)} 个")
    if len(missing)    > 0: issues.append(f"缺失字段 {len(missing)} 个")
    print(f"⚠ 存在问题：{'、'.join(issues)}")
print("=" * 50)